# Training: U-Net for Binary Burn Detection

Train the U-Net model on Sentinel-2 imagery for binary burn scar segmentation.

See [docs/TRAINING_PROCESS.md](../docs/TRAINING_PROCESS.md) for detailed information on loss functions, hyperparameters, troubleshooting, and device handling.

## Setup

In [ ]:
# Clone repo if not already present
import os
if not os.path.exists('RETINNA'):
    !git clone https://github.com/scerruti/RETINNA.git

# Add to path
import sys
sys.path.insert(0, '/content/RETINNA' if '/content' in os.getcwd() else 'RETINNA')

In [ ]:
# Install dependencies
%pip install -q torchgeo torch torchvision matplotlib ipywidgets

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from ipywidgets import IntSlider

from src.unet import UNet
from src.dataset import get_dataloaders
from src.device_utils import get_device, move_to_device

device = get_device()

## Data Loading

In [ ]:
# Setup Google Drive caching if on Colab
root = None
try:
    from src.colab_utils import setup_cabuaur_cached
    root = setup_cabuaur_cached()
    print("✓ Google Drive caching enabled")
except (ImportError, RuntimeError):
    print("Using default TorchGeo cache")

# Load dataloaders
print("\nLoading CaBuAr dataset...")
dataloaders = get_dataloaders(
    batch_size=4,
    num_workers=0,
    normalize=True,
    root=root
)

print(f"Train samples: {len(dataloaders['datasets']['train'])}")
print(f"Val samples: {len(dataloaders['datasets']['val'])}")
print(f"Test samples: {len(dataloaders['datasets']['test'])}")


## Model and Loss

In [ ]:
# pos_weight: higher = penalize false negatives (missed burns) more
# pos_weight > 1.0 encourages model to predict burns more aggressively  
# pos_weight = 1.5 means: penalize missing a burn 1.5× more than false positives
# This compensates for class imbalance by emphasizing burn detection

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, pos_weight=1.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.pos_weight = pos_weight
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

    def forward(self, predictions, targets):
        burned_logits = predictions[:, 1:2]
        bce_loss = self.bce(burned_logits, targets.float())

        probs = torch.softmax(predictions, dim=1)
        burned_prob = probs[:, 1:2]
        intersection = (burned_prob * targets).sum()
        union = (burned_prob + targets).sum()
        dice_loss = 1 - (2 * intersection + 1e-8) / (union + 1e-8)

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

def compute_iou(predictions, targets, threshold=0.5):
    with torch.no_grad():
        probs = torch.softmax(predictions, dim=1)
        burned_pred = (probs[:, 1] > threshold).float()
        targets_binary = targets.squeeze(1).float()
        intersection = (burned_pred * targets_binary).sum()
        union = (burned_pred + targets_binary).sum() - intersection
        iou = intersection / (union + 1e-8)
    return iou.item()

print("Creating model...")
model = UNet(in_channels=24, out_channels=2).to(device)
print(f"Model parameters: {model.get_parameter_count():,}")

criterion = BCEDiceLoss(pos_weight=1.5)
optimizer = Adam(model.parameters(), lr=0.0005)

## Training Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch_idx, batch in enumerate(train_loader):
        batch = move_to_device(batch, device)
        images = batch['image']
        masks = batch['mask']
        B, T, C, H, W = images.shape
        images = images.view(B, T * C, H, W)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if (batch_idx + 1) % 5 == 0:
            print(f"  Batch {batch_idx + 1}/{len(train_loader)}: Loss = {loss.item():.4f}")
    return total_loss / len(train_loader)

def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss, total_iou = 0.0, 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = move_to_device(batch, device)
            images = batch['image']
            masks = batch['mask']
            B, T, C, H, W = images.shape
            images = images.view(B, T * C, H, W)
            outputs = model(images)
            loss = criterion(outputs, masks)
            iou = compute_iou(outputs, masks)
            total_loss += loss.item()
            total_iou += iou
    return total_loss / len(val_loader), total_iou / len(val_loader)

## Training

In [ ]:
epochs_slider = IntSlider(value=5, min=1, max=50, step=1, description='Epochs:')
display(epochs_slider)

In [ ]:
# Setup checkpoint directories (local + Drive backup)
local_checkpoint_dir = Path('checkpoints_notebook')
local_checkpoint_dir.mkdir(exist_ok=True)

drive_checkpoint_dir = Path('/content/drive/MyDrive/RETINNA_checkpoints')
drive_checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Setup simple file logging
log_file = local_checkpoint_dir / 'training.log'

def log_message(msg):
    """Write to file and print"""
    print(msg)
    with open(log_file, 'a') as f:
        f.write(msg + '\n')
        f.flush()  # Force write to disk

num_epochs = epochs_slider.value
history = {'train_loss': [], 'val_loss': [], 'val_iou': []}
best_val_iou = 0.0

log_message("="*70)
log_message("Training Started")
log_message(f"Epochs: {num_epochs} | Device: {device}")
log_message(f"Local checkpoint dir: {local_checkpoint_dir}")
log_message(f"Drive backup dir: {drive_checkpoint_dir}")
log_message("="*70 + "\n")

for epoch in range(num_epochs):
    log_message(f"Epoch {epoch + 1}/{num_epochs}")
    
    train_loss = train_epoch(model, dataloaders['train'], criterion, optimizer, device)
    history['train_loss'].append(train_loss)
    log_message(f"  Train Loss: {train_loss:.4f}")

    val_loss, val_iou = validate(model, dataloaders['val'], criterion, device)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)
    log_message(f"  Val Loss: {val_loss:.4f}, Val IoU: {val_iou:.4f}")

    # Save checkpoint both locally and to Drive
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        
        # Save to local
        local_path = local_checkpoint_dir / 'best_model.pth'
        torch.save(model.state_dict(), local_path)
        
        # Backup to Google Drive
        import shutil
        drive_path = drive_checkpoint_dir / 'best_model.pth'
        shutil.copy(local_path, drive_path)
        
        log_message(f"  ★ Best model saved (IoU: {best_val_iou:.4f})")
        log_message(f"    Local: {local_path}")
        log_message(f"    Drive: {drive_path}")

log_message("="*70)
log_message(f"Training complete! Best Val IoU: {best_val_iou:.4f}")
log_message(f"Log file: {log_file}")
log_message("="*70)

# Verify Drive backup exists
if (drive_checkpoint_dir / 'best_model.pth').exists():
    print("\n✓ Checkpoint backed up to Google Drive (safe!)")
else:
    print("\n⚠ WARNING: Drive backup failed - checkpoint only on local")

## Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train', marker='o')
axes[0].plot(history['val_loss'], label='Val', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['val_iou'], marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].set_title('Validation IoU (Burned Class)')
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()